Script Overview:
- This script makes a sitemap for Canada as the sitemap was not available. 

In [3]:
import requests 
from bs4 import BeautifulSoup
from datetime import datetime
import xml.etree.ElementTree as ET
import os

In [ ]:

"""
This script scrapes all English subject-related article URLs from a specific Statistics Canada page:

Steps:
1. Sends a request to the specified URL and parses the HTML using BeautifulSoup.
2. Locates the table containing the article links.
3. Extracts all <a> tags from the table rows.
4. Converts partial URLs to full URLs using the base URL.
5. Stores the collected links in an XML format.
6. Saves the XML structure to 'urls.xml' 
"""

directory = "sitemaps_manual"
file_path = os.path.join(directory, "urls.xml")

URL = "https://www150.statcan.gc.ca/n1/dai-quo/ssi/homepage/rel-com/all_subjects-eng.htm"

# get the URL
page = requests.get(URL)
soup = BeautifulSoup(page.text, "html.parser")

# specify base url
base_url = "https://www150.statcan.gc.ca/n1"

# initialize html structure
root = ET.Element("urls")

# loop through the URL to find the next URL
body = soup.find("tbody")
url_sections = body.find_all("td")
for td in url_sections:
    link = td.find("a")
    if link and link.get("href"):
        href = link.get("href")
        full_url = href if href.startswith("https") else base_url + href

        # create xml element for each URL 
        url_element = ET.SubElement(root, "url")
        url_element.text = full_url


tree = ET.ElementTree(root)

with open(file_path, "wb") as xml_file:
    tree.write(xml_file, encoding = "utf-8", xml_declaration = True)

print(f"URLS saved to {file_path}")


URLS saved to sitemaps_manual/urls.xml


In [ ]:

"""
This script reads an XML file containing a list of URLs and filters out any URLs that are
archived. It does this by checking whether each page contains an HTML element
with the ID "archived".
"""

file_path = "sitemaps_manual/urls.xml"
tree = ET.parse(file_path)
root = tree.getroot()

# Loop through the URLs in the XML file
for url_element in root.findall("url"):
    full_url = url_element.text
    
    # Download the page content
    page_check = requests.get(full_url)
    page_soup = BeautifulSoup(page_check.text, "html.parser")
    
    # Check if the page contains an archival notice
    if page_soup.find(id="archived"):
        print(f"Skipping archived URL: {full_url}")
        # Remove the URL from the XML
        root.remove(url_element)
    else:
        print(f"Processing URL: {full_url}")
        # You can add further processing here

# Save the updated XML to file
tree.write(file_path, encoding="utf-8", xml_declaration=True)

print(f"Updated URLs saved to {file_path}")


